# Gold — Customer Dimension
One row per customer. CRM is the master; ERP fills in birth date, country and gender when the CRM does not know it.

→ `gold.dim_customers`

## Transformation (SQL)

In [ ]:
CATALOG = "workspace"

query = f"""
SELECT
    ROW_NUMBER() OVER (ORDER BY ci.customer_id)      AS customer_key,   -- surrogate key
    ci.customer_id,
    ci.customer_number,
    ci.first_name,
    ci.last_name,
    la.country,
    ci.marital_status,
    CASE
        WHEN ci.gender <> 'n/a' THEN ci.gender          -- CRM is the master for gender
        ELSE COALESCE(ca.gender, 'n/a')                 -- fall back to ERP
    END                                              AS gender,
    ca.birth_date,
    ci.created_date
FROM {CATALOG}.silver.crm_customers ci
LEFT JOIN {CATALOG}.silver.erp_customers ca
    ON ci.customer_number = ca.customer_number
LEFT JOIN {CATALOG}.silver.erp_customer_location la
    ON ci.customer_number = la.customer_number
"""

df = spark.sql(query)

## Sanity check

In [ ]:
df.limit(10).display()

## Write gold table

In [ ]:
df.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.gold.dim_customers")

In [ ]:
%sql
SELECT COUNT(*) AS rows FROM workspace.gold.dim_customers;